## Cell 1: setup and sample data

In [1]:
import re, math
from collections import Counter
from typing import Any, Dict
from dotenv import load_dotenv
import voyageai

load_dotenv()
voyage = voyageai.Client()

report = """# Annual Report

## Software Engineering
In Q4 the team resolved INC-2023-Q4-011, a deployment outage. Engineers fixed 247 bugs and shipped four releases.

## Cybersecurity Analysis
Incident INC-2023-Q4-011 involved a phishing attempt. The security team contained it within two hours and updated policies.

## Financial Analysis
Revenue grew 12% year over year. Operating costs stayed flat and the company ended the year with a healthy cash reserve.

## Methodology
Data was collected from internal systems and reviewed quarterly by an independent team.
"""

def chunk_by_section(text):
    return re.split(r"\n## ", text)

## Cell 2: the two indexes (same API)

Both take a document {"content": ...} through add_document and return (doc, score) pairs from search.

In [2]:
STOP_WORDS = {"a","an","the","and","or","of","to","in","on","with","what","which",
              "who","is","are","was","were","it","its","for","by","at","as","this",
              "that","happened"}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())
    return [w for w in words if w not in STOP_WORDS]

class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs, self.tokens = [], []

    def add_document(self, doc):
        self.docs.append(doc)
        self.tokens.append(tokenize(doc["content"]))

    def search(self, query_text, k=3):
        q_terms = tokenize(query_text)
        n = len(self.tokens)
        avg_len = sum(len(t) for t in self.tokens) / n
        df = Counter()
        for toks in self.tokens:
            for term in set(toks):
                df[term] += 1
        scored = []
        for doc, toks in zip(self.docs, self.tokens):
            tf = Counter(toks)
            score = 0.0
            for term in q_terms:
                if term in tf:
                    idf = math.log(1 + (n - df[term] + 0.5) / (df[term] + 0.5))
                    score += idf * tf[term] * (self.k1 + 1) / (
                        tf[term] + self.k1 * (1 - self.b + self.b * len(toks) / avg_len))
            scored.append((doc, score))
        return sorted(scored, key=lambda p: p[1], reverse=True)[:k]

class VectorIndex:
    def __init__(self):
        self.docs, self.vectors = [], []
        self._pending = []

    def add_document(self, doc):
        # Embeddings are computed in one batch later (saves requests)
        self._pending.append(doc)

    def _build(self):
        if self._pending:
            embs = voyage.embed([d["content"] for d in self._pending],
                                model="voyage-3-large", input_type="document").embeddings
            self.docs += self._pending
            self.vectors += embs
            self._pending = []

    def search(self, query_text, k=3):
        self._build()
        q = voyage.embed([query_text], model="voyage-3-large", input_type="query").embeddings[0]
        def dist(a, b):
            dot = sum(x*y for x, y in zip(a, b))
            return 1 - dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))
        scored = [(d, dist(q, v)) for d, v in zip(self.docs, self.vectors)]
        return sorted(scored, key=lambda p: p[1])[:k]

## Cell 3: the Retriever with reciprocal rank fusion

In [3]:
class Retriever:
    def __init__(self, *indexes):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document: Dict[str, Any]):
        for index in self._indexes:
            index.add_document(document)

    def search(self, query_text: str, k: int = 1, k_rrf: int = 60):
        scores = {}   # content -> RRF score
        docs = {}     # content -> document
        for index in self._indexes:
            results = index.search(query_text, k=len(index.docs) if index.docs else 10)
            for rank, (doc, _) in enumerate(results, start=1):
                key = doc["content"]
                docs[key] = doc
                scores[key] = scores.get(key, 0.0) + 1.0 / (k_rrf + rank)
        ranked = sorted(scores.items(), key=lambda p: p[1], reverse=True)[:k]
        return [(docs[key], score) for key, score in ranked]

Caveat: VectorIndex.docs fills only after the first search builds it, so I request up to 10 results for that call, which is enough here.

## Cell 4: check RRF against the lesson’s example

In [4]:
def rrf(ranks, k=1):
    return sum(1.0 / (k + r) for r in ranks)

print("Section 2:", round(rrf([1, 2]), 3))   # 0.833
print("Section 7:", round(rrf([2, 3]), 3))   # 0.583
print("Section 6:", round(rrf([3, 1]), 3))   # 0.75

Section 2: 0.833
Section 7: 0.583
Section 6: 0.75


## Cell 5: run the hybrid search (2 Voyage requests)

In [5]:
retriever = Retriever(VectorIndex(), BM25Index())
for chunk in chunk_by_section(report):
    retriever.add_document({"content": chunk})

results = retriever.search("What happened with INC-2023-Q4-011?", k=3, k_rrf=1)
for doc, score in results:
    print(round(score, 3), "-", doc["content"][:80].replace("\n", " "))

1.0 - Software Engineering In Q4 the team resolved INC-2023-Q4-011, a deployment outag
0.667 - Cybersecurity Analysis Incident INC-2023-Q4-011 involved a phishing attempt. The
0.45 - # Annual Report 
